# Phase 4: Property and Appraisal Review

This notebook prototypes the collateral specialist. It reconciles the application loan amount, purchase contract price, and appraisal value. A low appraisal is a review exception—not an autonomous lending decision.

```text
Phase 3 state → appraisal_review → PropertyAnalysis
```

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


In [ ]:
from underwriting_agent.document_layer import build_document_workflow
from underwriting_agent.borrower_analysis import build_borrower_workflow
from underwriting_agent.intake_packages import resolve_document_paths

def through_phase_3(loan_id):
    paths = resolve_document_paths(PDF_ROOT, loan_id)
    state = build_document_workflow().invoke({
        "loan_id": loan_id,
        "document_paths": [str(path) for path in paths],
        "workflow_status": "INTAKE",
    })
    return build_borrower_workflow().invoke(state)


## 1. Inspect the three collateral evidence sources

The specialist keeps all three document IDs so every value in the later review package can be traced back to its source PDF.

In [ ]:
from underwriting_agent.borrower_analysis import find_intake_document
from underwriting_agent.models import DocumentType

state = through_phase_3("BRK-90831")
for kind in [DocumentType.LOAN_APPLICATION, DocumentType.PURCHASE_CONTRACT, DocumentType.APPRAISAL]:
    parsed, intake = find_intake_document(state, kind)
    print(kind, parsed.document_id, intake.source_path.name)


## 2. Run the property subgraph

For a purchase, later LTV calculations must use the lower of contract price and appraised value.

In [ ]:
from underwriting_agent.property_analysis import build_property_workflow

property_workflow = build_property_workflow()
result = property_workflow.invoke(state)
result["property_analysis"].model_dump(mode="json")


## 3. Evaluate all appraisals

The Hayes and Nguyen packages should report negative variance and `LOW_APPRAISAL`.

In [ ]:
portfolio = []
for loan_id in INTAKE_REFERENCES:
    result = property_workflow.invoke(through_phase_3(loan_id))
    prop = result["property_analysis"]
    portfolio.append({"loan_id": loan_id, "purchase_price": prop.purchase_price, "appraised_value": prop.appraised_value, "variance": prop.value_variance, "exceptions": prop.exceptions})
portfolio


## Phase 4 handoff

The stable output is `PropertyAnalysis`. Phase 4B uses its address for isolated public-web research before Phase 5 calculates LTV.

## Optional production services: OpenAI and Pinecone

This phase intentionally remains deterministic. An OpenAI model may normalize messy source documents in Phase 2 and draft reviewer-facing prose in Phase 7, while borrower routing, income selection, asset exclusion, appraisal comparison, reconciliation, and exception rules remain typed Python logic. Pinecone is used only for guideline retrieval in Phase 5.